# FAMPNN Inverse Folding

This notebook demonstrates four functions of the FAMPNN tool. In the first section, we use `run_fampnn_sample` to design new amino acid sequences with simultaneous sidechain co-generation. In the second section, we use `run_fampnn_pack` to predict sidechain conformations for a fixed backbone and sequence. In the third section, we use `run_fampnn_score` to evaluate specific point mutations using full-atom log-likelihood ratios. In the fourth section, we use `run_fampnn_score_all_mutations` to exhaustively score every possible single amino acid substitution at every position.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("fampnn")
display_overview("fampnn")
display_docs_section("fampnn", "Background")

# FAMPNN

> FAMPNN (Full-Atom MPNN) is a deep learning model for protein sequence design that jointly models discrete amino acid identity and continuous sidechain conformation. Unlike backbone-only inverse folding models, FAMPNN generates both sequences and full-atom sidechain coordinates simultaneously using combined cross-entropy and diffusion loss. This module provides interfaces for *Sequence Sampling*, *Sidechain Packing*, *Mutation Scoring*, and *Exhaustive Mutation Scanning*.

## Background

**What does this tool do?**
FAMPNN solves the full-atom [inverse folding](https://en.wikipedia.org/wiki/Protein_design#Inverse_folding) problem: given a protein backbone structure, predict both the amino acid sequence and the [sidechain](https://en.wikipedia.org/wiki/Side_chain) conformations that are most compatible with the backbone. This goes beyond traditional inverse folding by also predicting atomic-level sidechain geometry.

**Why is this important?**
Full-atom design enables:

* More accurate protein engineering by modeling sidechain packing interactions
* Sidechain confidence scores (pSCE) for identifying uncertain regions
* Better mutation effect prediction using full structural context
* Direct generation of atomic models suitable for [molecular dynamics](https://en.wikipedia.org/wiki/Molecular_dynamics) or [docking](https://en.wikipedia.org/wiki/Molecular_docking)

**Scientific foundation:**
FAMPNN uses a two-component architecture:

1. **Iterative masked language modeling**: Sequence design via progressive unmasking (similar to MaskGIT), starting fully masked and revealing tokens iteratively.
2. **Per-token Euclidean [diffusion](https://en.wikipedia.org/wiki/Diffusion_model)**: Sidechain coordinates generated via variance-exploding EDM in local backbone reference frames.
3. **Predicted Sidechain Error (pSCE)**: A learned confidence metric predicting per-atom sidechain packing error in Angstroms.

## Available tools

In [2]:
display_available_tools("fampnn")

- **`run_fampnn_pack()`** — Pack protein sidechains using FAMPNN with per-atom confidence (pSCE)
- **`run_fampnn_sample()`** — Design protein sequences with full-atom sidechain co-generation using FAMPNN
- **`run_fampnn_score()`** — Score protein mutations with full-atom context using FAMPNN
- **`run_fampnn_score_all_mutations()`** — Score every possible single mutation at every position using FAMPNN

### `run_fampnn_sample`

FAMPNN designs new protein sequences that are predicted to fold into a target backbone structure by iteratively unmasking sequence positions while simultaneously denoising sidechain coordinates. Each designed residue comes with a predicted sidechain error (pSCE) score that quantifies confidence in the sidechain placement. You can optionally fix specific residue positions to preserve functional sites, and condition on known sidechain coordinates at selected positions to anchor the design around experimentally determined conformations.

In [3]:
from proto_tools.tools.inverse_folding.fampnn import (
    FAMPNNSampleInput,
    FAMPNNSampleConfig,
    FAMPNNStructureInput,
    run_fampnn_sample,
)
from proto_tools.entities.structures.examples import get_gfp_structure

In [4]:
# Display input docs
display_api_reference("fampnn", "input", "run_fampnn_sample")

# Load GFP as the target backbone
gfp_structure = get_gfp_structure()

# Design all chains
sample_input = FAMPNNSampleInput(
    inputs=[FAMPNNStructureInput(structure=gfp_structure)]
)

**Input** — `FAMPNNSampleInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `inputs` | `List[FAMPNNStructureInput]` | required | List of FAMPNN structure inputs, each containing a structure and optional chain\_ids/fixed\_positions/fixed\_sidechain\_positions. |
| `fixed_sidechain_positions` | `Dict[string, List[integer]]` |  | Optional dictionary mapping chain IDs to residue positions whose sidechain coordinates should be used as context during sampling/packing. Positions are 1-indexed. |
| `structure` | `Structure` | required | The protein structure. Accepts file path, PDB content string, or Structure object. Automatically converted to Structure. |
| `chain_ids` | `array` |  | Optional list of chain IDs to design. If None, all chains in the structure are designed. |
| `fixed_positions` | `Dict[string, List[integer]]` |  | Optional dictionary mapping chain IDs to residue positions to keep fixed during design. Positions are 1-indexed. |

In [5]:
# Display config docs
display_api_reference("fampnn", "config", "run_fampnn_sample")

# Configure sampling with custom parameters | see docs above for all fields
sample_config = FAMPNNSampleConfig(
    num_sequences_per_structure=2,
    temperature=0.1,
    num_steps=10,  # Use 100 for production runs
    seed=42,
    device="cuda",  # Change to "cpu" if no GPU available
)

# Fix specific residue positions to preserve functional sites
constrained_input = FAMPNNSampleInput(
    inputs=[FAMPNNStructureInput(
        structure=gfp_structure,
        chain_ids=["A"],
        fixed_positions={"A": [10, 15, 20, 25]},
        fixed_sidechain_positions={"A": [10, 15]},
    )]
)

**Config** — `FAMPNNSampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_variant` | `string` | `0.3` | FAMPNN checkpoint variant. '0.3' for sequence design (PDB-trained, 0.3A noise), '0.0' for sidechain packing (PDB-trained, 0.0A noise), '0.3\_cath' for mutation scoring (CATH-trained). |
| `num_steps` | `integer` | `100` | Number of iterative unmasking steps for sequence design. More steps yield higher quality but slower inference. 10 steps is sufficient for high self-consistency; 100 for best quality. |
| `seq_only` | `boolean` | `False` | If True, skip sidechain generation during sampling. |
| `repack_last` | `boolean` | `True` | If True, repack sidechains after final sequence is determined. |
| `psce_threshold` | `number` | `0.3` | Only condition on sidechains with predicted sidechain error below this threshold during iterative sampling. |
| `seed` | `integer` |  | Random seed to use for sampling. |
| `num_sequences_per_structure` | `integer` | `1` | Total number of sequences to generate per input structure. |
| `batch_size` | `integer` |  | Number of sequences to process simultaneously on GPU. Defaults to num\_sequences\_per\_structure. |
| `temperature` | `number` | `0.1` | Controls randomness in sampling from logits. |
| `excluded_amino_acids` | `array` |  | List of amino acids not allowed in the sequence. Not supported by FAMPNN (raises ValueError if set). |
| `scn_diffusion_steps` | `integer` | `50` | Number of sidechain diffusion denoising steps. |
| `scn_step_scale` | `number` | `1.5` | Step scale for sidechain diffusion (eta parameter). |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cuda` | Device to run the model on. Options include 'cuda' (NVIDIA GPU), 'cpu' (CPU execution), or specific GPU devices like 'cuda:0'. Defaults to 'cuda'. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [6]:
# Run the sampling function
result = run_fampnn_sample(constrained_input, sample_config)

Running run_fampnn_sample [00:00]

In [7]:
# Display output docs
display_api_reference("fampnn", "output", "run_fampnn_sample")

# Print designed sequences and per-residue pSCE scores
for i, designed in enumerate(result.designed_sequences):
    for j, seq in enumerate(designed.sequences):
        display_seq = f"{seq[:60]}..." if len(seq) > 60 else seq
        mean_psce = sum(designed.psce[j]) / len(designed.psce[j])
        print(f"Structure {i}, Design {j + 1}: {display_seq}")
        print(f"  Length: {len(seq)} residues, Mean pSCE: {mean_psce:.3f} A")

**Output** — `InverseFoldingOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `designed_sequences` | `List[DesignedSequences]` | required | List of DesignedSequences objects, one for each input structure. The order matches the input structures order. |
| `sequences` | `List[string]` | required | Designed amino acid sequences in single-letter code. |

Structure 0, Design 1: SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLV...
  Length: 227 residues, Mean pSCE: 1.286 A
Structure 0, Design 2: SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLV...
  Length: 227 residues, Mean pSCE: 1.293 A


### `run_fampnn_pack`

Given a backbone structure and its sequence, FAMPNN predicts sidechain coordinates using per-token Euclidean diffusion. This is useful when you have a designed or known sequence but need atomic-level sidechain placement for downstream analysis such as molecular dynamics or ligand docking. The B-factor column in the output PDB contains per-atom pSCE values, so coloring by B-factor in a molecular viewer directly shows sidechain confidence.

In [8]:
from proto_tools.tools.inverse_folding.fampnn import (
    FAMPNNPackInput,
    FAMPNNPackConfig,
    run_fampnn_pack,
)

In [9]:
# Display input docs
display_api_reference("fampnn", "input", "run_fampnn_pack")

# Pack sidechains for the GFP structure
pack_input = FAMPNNPackInput(
    inputs=[FAMPNNStructureInput(structure=gfp_structure)]
)

**Input** — `FAMPNNPackInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `inputs` | `List[FAMPNNStructureInput]` | required | List of structure inputs for sidechain packing. |
| `fixed_sidechain_positions` | `Dict[string, List[integer]]` |  | Optional dictionary mapping chain IDs to residue positions whose sidechain coordinates should be used as context during sampling/packing. Positions are 1-indexed. |
| `structure` | `Structure` | required | The protein structure. Accepts file path, PDB content string, or Structure object. Automatically converted to Structure. |
| `chain_ids` | `array` |  | Optional list of chain IDs to design. If None, all chains in the structure are designed. |
| `fixed_positions` | `Dict[string, List[integer]]` |  | Optional dictionary mapping chain IDs to residue positions to keep fixed during design. Positions are 1-indexed. |

In [10]:
# Display config docs
display_api_reference("fampnn", "config", "run_fampnn_pack")

# Configure packing | see docs above for all fields
pack_config = FAMPNNPackConfig(
    num_samples_per_structure=3,
    batch_size=3,
    seed=42,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `FAMPNNPackConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_variant` | `string` | `0.0` | Checkpoint variant. '0.0' recommended for best packing accuracy. |
| `num_samples_per_structure` | `integer` | `1` | Number of packing samples per input structure. |
| `batch_size` | `integer` | `1` | Number of samples to process simultaneously on GPU. |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `scn_diffusion_steps` | `integer` | `50` | Number of sidechain diffusion denoising steps. |
| `scn_step_scale` | `number` | `1.5` | Step scale for sidechain diffusion. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cuda` | Device to run on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [11]:
# Run the packing function
pack_result = run_fampnn_pack(pack_input, pack_config)

Running run_fampnn_pack [00:00]

In [12]:
# Display output docs
display_api_reference("fampnn", "output", "run_fampnn_pack")

# Print mean pSCE for each packing sample
for i, (pdbs, psce_list) in enumerate(zip(pack_result.packed_structures, pack_result.psce)):
    print(f"Structure {i}: {len(pdbs)} packing samples")
    for j, psce in enumerate(psce_list):
        mean_psce = sum(psce) / len(psce)
        print(f"  Sample {j + 1}: mean pSCE = {mean_psce:.3f} A")

**Output** — `FAMPNNPackingResult`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `packed_structures` | `List[array]` | required | List of lists of PDB strings with packed sidechain coordinates. Outer list corresponds to input structures, inner list to packing samples. B-factor column contains per-atom pSCE. |
| `psce` | `List[array]` | required | Per-residue predicted sidechain error (Angstroms) for each sample. |

Structure 0: 3 packing samples
  Sample 1: mean pSCE = 1.280 A
  Sample 2: mean pSCE = 1.279 A
  Sample 3: mean pSCE = 1.279 A


### `run_fampnn_score`

FAMPNN can score specific point mutations by computing log-likelihood ratios in full-atom context. Mutations are specified in the format `WT_RESIDUE + 1_INDEXED_POSITION + MUTANT_RESIDUE` (for example, `"S1V"` means serine to valine at position 1). Multiple simultaneous mutations are joined with colons: `"S1V:G5L"`. Positive scores indicate that the mutation is favored over wild-type. Use the special string `"wt"` to include the wild-type as a reference point, which always scores 0.0.

In [13]:
from proto_tools.tools.inverse_folding.fampnn import (
    FAMPNNScoreInput,
    FAMPNNScoreConfig,
    MutationInput,
    run_fampnn_score,
)

In [14]:
# Display input docs
display_api_reference("fampnn", "input", "run_fampnn_score")

# Score specific mutations (1-indexed positions)
score_input = FAMPNNScoreInput(
    inputs=[MutationInput(
        structure=gfp_structure,
        mutations=["S1V", "K2A", "wt"],
    )]
)

**Input** — `FAMPNNScoreInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `inputs` | `List[MutationInput]` | required | List of MutationInput objects, each containing a structure and mutations to score. |
| `structure` | `Structure` | required | Protein structure to evaluate mutations against. |
| `mutations` | `List[string]` | required | List of mutation strings. Each mutation uses the format '\\<1-indexed\_position>\' with single-letter amino acid codes. Multiple simultaneous mutations are joined with colons: 'N1P:N2R'. |

In [15]:
# Display config docs
display_api_reference("fampnn", "config", "run_fampnn_score")

# Configure mutation scoring | see docs above for all fields
score_config = FAMPNNScoreConfig(
    seed=42,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `FAMPNNScoreConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_variant` | `string` | `0.3_cath` | Checkpoint variant. '0.3\_cath' recommended for scoring. |
| `batch_size` | `integer` | `16` | Number of mutations to score simultaneously on GPU. |
| `seq_only` | `boolean` | `False` | If True, score without sidechain context (backbone-only). |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `scn_diffusion_steps` | `integer` | `50` | Number of sidechain diffusion denoising steps. |
| `scn_step_scale` | `number` | `1.5` | Step scale for sidechain diffusion. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cuda` | Device to run on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [16]:
# Run the scoring function
score_result = run_fampnn_score(score_input, score_config)

Running run_fampnn_score [00:00]

In [17]:
# Display output docs
display_api_reference("fampnn", "output", "run_fampnn_score")

# Print scores for each mutation
for mut, score in zip(score_result.results[0].mutations, score_result.results[0].scores):
    label = " (wild-type)" if mut == "wt" else ""
    print(f"{mut}: {score:+.4f}{label}")

**Output** — `FAMPNNScoreOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `List[MutationScoreResult]` | required | List of MutationScoreResult objects, one per input structure. |
| `mutations` | `List[string]` | required | Mutation strings that were scored. |
| `scores` | `List[number]` | required | Log-likelihood ratio scores for each mutation. Positive = mutation is more likely than wild-type. |

S1V: -4.9167
K2A: -0.1259
wt: +0.0000 (wild-type)


### `run_fampnn_score_all_mutations`

For a comprehensive mutational landscape, FAMPNN can exhaustively score every possible single amino acid substitution at every position in the protein. The result is a position-by-residue dictionary of log-likelihood ratios that can be used to identify stabilizing mutations, map evolutionary constraints, or guide directed evolution experiments.

In [18]:
from proto_tools.tools.inverse_folding.fampnn import (
    FAMPNNScoreAllMutationsInput,
    FAMPNNScoreAllMutationsConfig,
    run_fampnn_score_all_mutations,
)

In [19]:
# Display input docs
display_api_reference("fampnn", "input", "run_fampnn_score_all_mutations")

# Score all single mutations for the GFP structure
all_mut_input = FAMPNNScoreAllMutationsInput(inputs=[gfp_structure])

**Input** — `FAMPNNScoreAllMutationsInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `inputs` | `List[Structure]` | required | List of structures to score all mutations for. |
| `structure` | `string` | required | Raw structure content in PDB or CIF format. |
| `structure_format` | `string` |  | Format of the content string (auto-detected if omitted). |
| `b_factor_type` | `BFactorType` | `unspecified` | What the B-factor column represents. |
| `source` | `string` |  | Optional source identifier (filepath or tool name). |
| `metrics` | `StructureMetrics` |  | Associated metrics (e.g., pLDDT, pTM scores, per-chain lists, pairwise matrices). None values are stripped at construction. |

In [20]:
# Display config docs
display_api_reference("fampnn", "config", "run_fampnn_score_all_mutations")

# Configure exhaustive scoring | see docs above for all fields
all_mut_config = FAMPNNScoreAllMutationsConfig(
    seed=42,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `FAMPNNScoreAllMutationsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_variant` | `string` | `0.3_cath` | Checkpoint variant. '0.3\_cath' recommended for scoring. |
| `batch_size` | `integer` | `16` | Number of positions to score simultaneously on GPU. |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cuda` | Device to run on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [21]:
# Run the exhaustive scoring function
all_mut_result = run_fampnn_score_all_mutations(all_mut_input, all_mut_config)

Running run_fampnn_score_all_mutations [00:00]

In [22]:
# Display output docs
display_api_reference("fampnn", "output", "run_fampnn_score_all_mutations")

# Find the top beneficial mutations across all positions
scores = all_mut_result.results[0].scores
all_scores = []
for pos_label, pos_scores in scores.items():
    wt_residue = pos_label[-1]
    for mut_aa, score in pos_scores.items():
        if mut_aa != wt_residue:
            all_scores.append((pos_label, mut_aa, score))

all_scores.sort(key=lambda x: x[2], reverse=True)

print(f"Scored {len(scores)} positions x 20 amino acids")
print()
print("Top 5 beneficial mutations:")
for pos, aa, s in all_scores[:5]:
    print(f"  {pos} -> {aa}: {s:+.4f}")

**Output** — `FAMPNNScoreAllMutationsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `List[AllMutationsScoreResult]` | required | List of AllMutationsScoreResult objects, one per input structure. |
| `scores` | `Dict[string, Dict[string, number]]` | required | Dictionary mapping position labels (e.g., '1A' for position 1, wild-type Ala) to dictionaries of \{mutant\_residue: score}. Scores are log-likelihood ratios (positive = favored over wild-type). |

Scored 227 positions x 20 amino acids

Top 5 beneficial mutations:
  217L -> E: +4.3040
  24H -> K: +4.2155
  167I -> V: +4.1633
  162K -> T: +4.0760
  143H -> S: +3.7119


## Export Results

Designed sequences can be exported to FASTA or JSON format. Packed sidechain structures can be exported as PDB files. Mutation scoring results can be exported to CSV or JSON format for downstream analysis.

In [23]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="fampnn_sequences", export_path=output_dir, file_format="fasta")
print(f"Designed sequences exported to {output_dir / 'fampnn_sequences.fasta'}")

pack_result.export(name="fampnn_packed", export_path=output_dir, file_format="pdb")
print(f"Packed structures exported to {output_dir / 'fampnn_packed/'}")

score_result.export(name="fampnn_scores", export_path=output_dir, file_format="csv")
print(f"Mutation scores exported to {output_dir / 'fampnn_scores.csv'}")

all_mut_result.export(name="fampnn_all_mutations", export_path=output_dir, file_format="csv")
print(f"All-mutation scores exported to {output_dir / 'fampnn_all_mutations/'}")

Designed sequences exported to example_output/fampnn_sequences.fasta
Packed structures exported to example_output/fampnn_packed
Mutation scores exported to example_output/fampnn_scores.csv
All-mutation scores exported to example_output/fampnn_all_mutations
